# Preparar paquetes de medición Android

Orden: 1. Conectar Drive → 2. Reconstruir el test de caña → 3. Cargar el exportador → 4. Crear el paquete de caña → 5. Consultar el estado de arroz. Puedes ejecutar de arriba abajo. No se entrena ni se cambia ninguna partición.

Se reconstruyen únicamente las 412 imágenes originales de test de caña mediante la trazabilidad guardada. Se comprueban sus hashes y el manifiesto del experimento antes de exportar. El paquete ocupa unos 248 MB; se necesita espacio libre en Drive y Colab.

## 1. Conectar Drive y comprobar ubicaciones

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/CNN_Lista')
print('Drive:', BASE, 'existe:', BASE.is_dir())
print('ZIP caña:', (BASE / 'Dataset_cana.zip').is_file())
print('Experimento caña:', (BASE / 'Resultados/cana/20261112T143413_595888Z_d6a38c7e').is_dir())
print('Partición caña:', (BASE / 'Particiones/cana_rafagas10_0ef893c2').is_dir())


## 2. Reconstruir únicamente test de caña

Esta celda lee Dataset_cana.zip desde Drive, copia solo las fotos de test al almacenamiento temporal de Colab y verifica cada SHA-256. Si una ejecución se interrumpe, puedes repetirla: reconocerá las fotos ya verificadas. Debe terminar con 412 y True. No borra ni modifica el ZIP original.

In [ ]:
from pathlib import Path, PurePosixPath
import csv
import hashlib
import json
import shutil
import zipfile

BASE = Path('/content/drive/MyDrive/CNN_Lista')
PARTICION = BASE / 'Particiones/cana_rafagas10_0ef893c2'
EXPERIMENTO = BASE / 'Resultados/cana/20261112T143413_595888Z_d6a38c7e'
ZIP_ORIGINAL = BASE / 'Dataset_cana.zip'
MANIFIESTO_ORIGINAL = EXPERIMENTO / 'manifest_experimento.csv'
TRAZABILIDAD = PARTICION / 'trazabilidad.csv'
RAIZ_DATOS = Path('/content/cnn_datos/cana_rafagas_10s')


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open('rb') as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()


for ruta in (ZIP_ORIGINAL, MANIFIESTO_ORIGINAL, TRAZABILIDAD,
             EXPERIMENTO / 'experimento.json', EXPERIMENTO / 'test_final.json'):
    if not ruta.is_file():
        raise FileNotFoundError(f'Falta: {ruta}')

meta = json.loads((EXPERIMENTO / 'experimento.json').read_text(encoding='utf-8'))
prueba_final = json.loads((EXPERIMENTO / 'test_final.json').read_text(encoding='utf-8'))
hash_manifest = sha256_archivo(MANIFIESTO_ORIGINAL)
if hash_manifest != meta['manifest_sha256'] or hash_manifest != prueba_final['manifest_sha256']:
    raise ValueError('El manifiesto no corresponde al experimento y test final de caña.')

with MANIFIESTO_ORIGINAL.open(encoding='utf-8-sig', newline='') as f:
    filas_test = [r for r in csv.DictReader(f, delimiter=';') if r['particion'] == 'test']

with TRAZABILIDAD.open(encoding='utf-8-sig', newline='') as f:
    mapa = {r['archivo_nuevo']: r['archivo_original']
            for r in csv.DictReader(f, delimiter=';')}

if len(filas_test) != 412 or len({r['archivo'] for r in filas_test}) != 412:
    raise ValueError('El manifiesto debe contener 412 imágenes únicas de test.')
if any(r['archivo'] not in mapa for r in filas_test):
    raise ValueError('Faltan archivos de test en la trazabilidad guardada.')

RAIZ_DATOS.mkdir(parents=True, exist_ok=True)
con_zip = 0
ya_presentes = 0
with zipfile.ZipFile(ZIP_ORIGINAL) as z:
    for numero, fila in enumerate(filas_test, 1):
        relativo = PurePosixPath(fila['archivo'])
        original = PurePosixPath(mapa[fila['archivo']])
        clase = fila['clase']
        if (clase not in ('sano', 'roya') or
            int(fila['etiqueta']) != {'sano': 0, 'roya': 1}[clase] or
            relativo.is_absolute() or '..' in relativo.parts or
            original.is_absolute() or '..' in original.parts or
            not fila['archivo'].startswith(f'test/{clase}/')):
            raise ValueError(f'Ruta o clase inválida: {fila["archivo"]}')

        destino = RAIZ_DATOS.joinpath(*relativo.parts)
        if destino.exists():
            if sha256_archivo(destino) != fila['sha256_archivo']:
                raise ValueError(f'Archivo existente no coincide: {destino}')
            ya_presentes += 1
            continue

        nombre_zip = 'cana/' + original.as_posix()
        destino.parent.mkdir(parents=True, exist_ok=True)
        temporal = destino.with_name(destino.name + '.part')
        try:
            with z.open(nombre_zip) as entrada, temporal.open('wb') as salida:
                shutil.copyfileobj(entrada, salida, length=1024 * 1024)
            if sha256_archivo(temporal) != fila['sha256_archivo']:
                raise ValueError(f'La foto no coincide con el manifiesto: {relativo}')
            temporal.replace(destino)
        finally:
            if temporal.exists():
                temporal.unlink()
        con_zip += 1
        if numero % 50 == 0:
            print(f'Procesadas: {numero}/412')

manifiesto_destino = RAIZ_DATOS / 'manifest.csv'
if manifiesto_destino.exists() and sha256_archivo(manifiesto_destino) != hash_manifest:
    raise ValueError(f'Ya existe otro manifiesto en {manifiesto_destino}')
shutil.copyfile(MANIFIESTO_ORIGINAL, manifiesto_destino)
assert sha256_archivo(manifiesto_destino) == hash_manifest
print('Caña lista:', RAIZ_DATOS)
print('Imágenes nuevas:', con_zip, '| ya verificadas:', ya_presentes)
print('Fotos de test verificadas:', con_zip + ya_presentes)
print('Manifiesto coincide:', sha256_archivo(manifiesto_destino) == hash_manifest)


## 3. Definir el exportador

Ejecuta la siguiente celda. Solo define funciones, por lo que no tiene salida. Prepara cada imagen con el mismo preprocesamiento EXIF, RGB, resize_with_pad y normalización que se usó en Colab, y conserva la puntuación de referencia del TFLite original.

In [ ]:
"""Exporta tensores EXACTOS de test a un paquete CBENCH1, sin reentrenar.

Uso desde Colab: ejecutar la celda de configuración del cuaderno adjunto.
Requiere la carpeta de la partición reconstruida y el experimento final.
"""
from __future__ import annotations

import csv
import hashlib
import json
import shutil
import struct
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf

MODELOS = {
    'arroz': ('0728e2ddedc0db0959870ef420c204df3094fd780ef520c0d3fb2b10a9090040',
              0.6501769423484802, 219, {'sano': 0, 'piricularia': 1}),
    'cana': ('9042f0dffc63a6c143fd64a8d36a34ab25bc96cb74b21e51df35d7e88a368bec',
             0.6862450242042542, 412, {'sano': 0, 'roya': 1}),
}
PREPROCESAMIENTO = 'rgb_exif_tf_bilinear_pad_224_div255_v1'
TENSOR_BYTES = 224 * 224 * 3 * 4


def sha_archivo(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()


def preparar(path):
    # Mismo orden y mismas funciones que en 01_Entrenamiento_CNN.ipynb.
    with Image.open(path) as foto:
        rgb = np.asarray(ImageOps.exif_transpose(foto).convert('RGB'), dtype=np.uint8)
    tensor = tf.image.resize_with_pad(tf.cast(rgb, tf.float32), 224, 224,
                                       method='bilinear', antialias=True) / 255.0
    result = np.asarray(tensor.numpy(), dtype='<f4')
    assert result.shape == (224, 224, 3) and result.nbytes == TENSOR_BYTES
    return result


def exportar(cultivo, raiz_datos, experimento, destino):
    if cultivo not in MODELOS:
        raise ValueError('Cultivo no reconocido.')
    sha_esperado, umbral, cantidad, clases = MODELOS[cultivo]
    raiz_datos, experimento, destino = map(Path, (raiz_datos, experimento, destino))
    modelo = experimento / 'modelo_float32.tflite'
    meta = json.loads((experimento / 'metadatos_modelo.json').read_text(encoding='utf-8'))
    fin = json.loads((experimento / 'test_final.json').read_text(encoding='utf-8'))
    manifest = raiz_datos / 'manifest.csv'
    if sha_archivo(modelo) != sha_esperado or meta['sha256_tflite'] != sha_esperado:
        raise ValueError('El modelo no coincide con los pesos fijos de este prototipo.')
    if meta['umbral_positivo_inclusivo'] != umbral or meta['preprocesamiento']['version'] != PREPROCESAMIENTO:
        raise ValueError('Metadatos de umbral o preprocesamiento incompatibles.')
    if fin['modelo_sha256'] != sha_esperado or fin['manifest_sha256'] != sha_archivo(manifest):
        raise ValueError('El test final corresponde a otra versión del modelo o inventario.')
    with manifest.open(encoding='utf-8-sig', newline='') as f:
        filas = [r for r in csv.DictReader(f, delimiter=';') if r['particion'] == 'test']
    if len(filas) != cantidad or len({r['archivo'] for r in filas}) != cantidad:
        raise ValueError('Falta alguna foto de test o hay duplicados.')
    if {r['etiqueta'] for r in filas} != {'0', '1'}:
        raise ValueError('El test debe contener ambas clases.')

    interprete = tf.lite.Interpreter(model_path=str(modelo), num_threads=1)
    interprete.allocate_tensors()
    entrada = interprete.get_input_details()[0]
    salida = interprete.get_output_details()[0]
    if tuple(entrada['shape']) != (1, 224, 224, 3) or entrada['dtype'] != np.float32:
        raise ValueError('Tensor de entrada incompatible.')
    if tuple(salida['shape']) != (1, 1) or salida['dtype'] != np.float32:
        raise ValueError('Tensor de salida incompatible.')

    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = Path(tempfile.mkdtemp(prefix='cbench_', dir=str(destino.parent)))
    datos = temporal / 'tensores.bin'
    try:
        items = []
        with datos.open('wb') as out:
            for i, fila in enumerate(filas, 1):
                relativo = Path(fila['archivo'])
                if (relativo.is_absolute() or '..' in relativo.parts or
                    not fila['archivo'].startswith(f"test/{fila['clase']}/") or
                    fila['clase'] not in clases or int(fila['etiqueta']) != clases[fila['clase']]):
                    raise ValueError(f'Ruta o etiqueta inválida: {relativo}')
                original = (raiz_datos / relativo).resolve()
                if not original.is_relative_to(raiz_datos.resolve()) or not original.is_file():
                    raise FileNotFoundError(original)
                if sha_archivo(original) != fila['sha256_archivo']:
                    raise ValueError(f'La imagen no coincide con el inventario: {relativo}')
                tensor = preparar(original)
                interprete.set_tensor(entrada['index'], tensor[None, ...])
                interprete.invoke()
                score = float(interprete.get_tensor(salida['index'])[0][0])
                bruto = tensor.tobytes(order='C')
                out.write(bruto)
                items.append({'archivo': fila['archivo'], 'etiqueta': int(fila['etiqueta']),
                              'sha256_tensor': hashlib.sha256(bruto).hexdigest(),
                              'score_referencia': score})
                if i % 50 == 0: print(f'{cultivo}: {i}/{cantidad}')
        cabecera = json.dumps({
            'schema': 1, 'cultivo': cultivo, 'particion': 'test',
            'modelo_sha256': sha_esperado, 'manifest_sha256': sha_archivo(manifest),
            'preprocesamiento': PREPROCESAMIENTO, 'umbral': umbral,
            'tensor_bytes': TENSOR_BYTES, 'items': items,
        }, ensure_ascii=False, separators=(',', ':'), allow_nan=False).encode('utf-8')
        if len(cabecera) > 1000000:
            raise ValueError('Cabecera demasiado grande.')
        tmp = destino.with_suffix(destino.suffix + '.part')
        with tmp.open('wb') as out, datos.open('rb') as src:
            out.write(b'CBENCH1\n')
            out.write(struct.pack('<I', len(cabecera)))
            out.write(cabecera)
            shutil.copyfileobj(src, out, length=1024 * 1024)
        tmp.replace(destino)
        print(f'Paquete verificado: {destino} ({destino.stat().st_size:,} bytes)')
        return destino
    finally:
        shutil.rmtree(temporal)


## 4. Exportar solo caña

Esta celda genera Paquetes_medicion/cana_test.cbm en Drive. Puede tardar varios minutos. El paquete contiene tensores Float32 y el inventario de las 412 fotos de test. No mide tiempos Android.

In [ ]:
DESTINO = BASE / 'Paquetes_medicion'
PAQUETE_CANA = exportar(
    'cana',
    Path('/content/cnn_datos/cana_rafagas_10s'),
    BASE / 'Resultados/cana/20261112T143413_595888Z_d6a38c7e',
    DESTINO / 'cana_test.cbm',
)
print('Archivo:', PAQUETE_CANA)
print('Tamaño (MB):', round(PAQUETE_CANA.stat().st_size / 1_000_000, 2))


## 5. Arroz: comprobar requisitos y exportar cuando estén disponibles

El experimento y el conjunto de fotos de arroz no estaban en las rutas anteriores de este Drive. La celda indica qué falta y solo exporta cuando las rutas estén completas. No uses una partición nueva ni cambies el test original de arroz. Cuando localices el ZIP de resultados y las fotos originales, reconstruiremos la ruta exacta.

In [ ]:
DATOS_ARROZ = Path('/content/cnn_datos/arroz')
EXPERIMENTO_ARROZ = BASE / 'Resultados/arroz/20261110T201636_634345Z_dfd6cd8a'
print('Arroz: manifest.csv:', (DATOS_ARROZ / 'manifest.csv').is_file())
print('Arroz: test/:', (DATOS_ARROZ / 'test').is_dir())
print('Arroz: modelo:', (EXPERIMENTO_ARROZ / 'modelo_float32.tflite').is_file())
print('Arroz: test_final.json:', (EXPERIMENTO_ARROZ / 'test_final.json').is_file())
if all([(DATOS_ARROZ / 'manifest.csv').is_file(),
        (DATOS_ARROZ / 'test').is_dir(),
        (EXPERIMENTO_ARROZ / 'modelo_float32.tflite').is_file(),
        (EXPERIMENTO_ARROZ / 'test_final.json').is_file()]):
    PAQUETE_ARROZ = exportar('arroz', DATOS_ARROZ, EXPERIMENTO_ARROZ,
                             BASE / 'Paquetes_medicion/arroz_test.cbm')
    print('Arroz listo:', PAQUETE_ARROZ)
else:
    print('Arroz pendiente. Esta celda no modificó archivos de arroz.')
